In [5]:
import pandas as pd
import numpy as np

# Read prior data
prior_data = pd.read_csv('priors/cultivar_requirements.csv')

# Calculate mean and SD per species for requirements
species_stats = prior_data.groupby('Species')[['CP', 'GDH', 'CH', 'CU']].agg(['mean', 'std']).reset_index()
species_stats.columns = ['Species', 'CP_mean', 'CP_std', 'GDH_mean', 'GDH_std', 'CH_mean', 'CH_std', 'CU_mean', 'CU_std']

print("Species-level statistics:")
print(species_stats)
print("\n")

# Save species-level statistics for hierarchical model
species_stats.to_csv('priors/prior_species_stats_hierarchical.csv', index=False)
print("Species-level priors saved to priors/prior_species_stats_hierarchical.csv")
print("\n")

prior_all = prior_data[prior_data['CP'].notna()].copy()
prior_all = prior_all.merge(species_stats, on='Species', how='left')


prior_all['CP_standardized'] = (prior_all['CP'] - prior_all['CP_mean']) / prior_all['CP_std']
prior_all['GDH_standardized'] = (prior_all['GDH'] - prior_all['GDH_mean']) / prior_all['GDH_std']

prior_all_rounded = prior_all.round(2)
prior_all_rounded.to_csv('priors/prior_allcultivars_hierarchical.csv', index=False)



# Read adamedor data for cultivars of interest
adamedor_data = pd.read_csv('phenology/adamedor_sub.csv')
cultivars_of_interest = adamedor_data['cultivar'].unique()

# Filter prior data for cultivars in adamedor and with CP data
prior_data = prior_data[(prior_data['Cultivar'].isin(cultivars_of_interest)) & (prior_data['CP'].notna())].copy()

# Merge with species statistics
prior_data = prior_data.merge(species_stats, on='Species', how='left')

# Standardize CP and GDH using species-level mean and SD
prior_data['CP_standardized'] = (prior_data['CP'] - prior_data['CP_mean']) / prior_data['CP_std']
prior_data['GDH_standardized'] = (prior_data['GDH'] - prior_data['GDH_mean']) / prior_data['GDH_std']

print("Filtered prior data with standardized values:")
print(prior_data[['Species', 'Cultivar', 'CP', 'CP_standardized', 'GDH', 'GDH_standardized']])

# Round numeric values to 2 decimal places
prior_data_rounded = prior_data.round(2)

# Save filtered prior data for hierarchical model
prior_data_rounded.to_csv('priors/prior_adamedor_hierarchical.csv', index=False)
print("Filtered prior data saved to priors/prior_hierarchical.csv")

Species-level statistics:
            Species    CP_mean     CP_std     GDH_mean      GDH_std  \
0            Almond  20.123125  13.016547  7282.462121  1398.086628   
1             Apple  58.461842  15.576298  8666.597015  2819.362286   
2           Apricot  50.937778  11.337313  4036.727273  1519.887093   
3     Japanese Plum  40.354615   9.240577  6494.507692  1465.260148   
4  Japanese apricot  60.060000  13.332028  1393.760000   394.020470   
5              Pear  47.456250   1.220912  7037.812500   323.153466   
6       Sour Cherry        NaN        NaN  6130.000000          NaN   
7      Sweet Cherry  51.427586  12.722921  8758.361538  2782.007859   

      CH_mean      CH_std      CU_mean      CU_std  
0  147.780220  138.313637   242.155963  291.107310  
1  972.632353  200.836208  1088.684211  348.585257  
2  827.195402  307.195788   996.480392  241.583585  
3  462.784722  136.422083   686.176923  179.474612  
4  670.375000  374.760471   964.000000  364.153814  
5  752.000000   